## 🏗️ LeetCode 973: K Closest Points to Origin
---

<div style="padding: 15px; border-left: 8px solid #4CAF50; background-color: #e8f5e9; color: #1b5e20; border-radius: 4px;">
    <strong>The Core Insight:</strong> We need the k CLOSEST points — not all n points sorted. A max-heap of size k keeps the k closest seen so far. The root (farthest of the k closest) is our eviction threshold: any new point farther than the root gets discarded.
</div>

### 🛠️ The Mathematical Model

Euclidean distance²: `x² + y²` (skip sqrt — monotone transformation preserves ordering). Maintain a max-heap of size k on distance². Root = farthest of top k = eviction boundary.

$$d^2 = x^2 + y^2 \quad \text{(no sqrt needed — monotone, preserves ordering)}$$

---

### 📋 Problem

Given a list of points on a plane and integer k, return the k closest points to the origin (0, 0). Distance formula: Euclidean. No sorting requirement for the output.

**Example 1:**
```
Input:  points = [[1,3],[-2,2]], k = 1
Output: [[-2,2]]
```

**Example 2:**
```
Input:  points = [[3,3],[5,-1],[-2,4]], k = 2
Output: [[3,3],[-2,4]]
```

**Constraints:** 1 ≤ k ≤ points.length ≤ 10⁴ | -10⁴ ≤ xi, yi ≤ 10⁴

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Shrinking Circle</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"Draw a circle around the origin. Start large. As you scan each point, if it fits inside the current k-point circle, it enters. If not, it's evicted. The circle shrinks as better points arrive."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Max-heap root = current circle radius. New point closer than root → enters, root leaves.</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>VIP Nearest Neighbors</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"In a store, the k closest customers get a coupon. As new customers arrive, check if they're closer than the current farthest VIP customer. If yes, they get the coupon and the farthest VIP loses it."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Max-heap of size k on distance — root is farthest VIP = eviction candidate</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> Sorting all n points by distance is O(n log n). A max-heap of size k processes each point in O(log k) for a total of O(n log k) — better when k << n.
</div>

## 🐢 Approach 1: Brute Force — $O(n \log n)$

In [ ]:
def kClosest_brute(points, k):
    """
    Brute Force: Sort all points by distance, take first k
    Time: O(n log n) | Space: O(n) for sort
    """
    points.sort(key=lambda p: p[0]**2 + p[1]**2)
    return points[:k]


print(kClosest_brute([[1,3],[-2,2]], 1))          # Expected: [[-2, 2]]
print(kClosest_brute([[3,3],[5,-1],[-2,4]], 2))   # Expected: [[3,3],[-2,4]]

## 🔬 Complexity Analysis: $O(n \log n)$ vs. $O(n \log k)$

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> We don't need all n points sorted — we only need the k closest. A max-heap of size k is our filter: it keeps the k closest seen so far, with the farthest-of-the-closest as the eviction boundary. New points farther than the boundary are rejected in O(log k).
</div>

---

## 📉 Why Brute Force Fails: The $O(n \log n)$ Trap

Sort computes relative order for ALL n points. When k = 10 and n = 10,000, sorting ranks 9,990 irrelevant points.

| Input Type | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **k = 1, n = 10,000** | O(n log n) = ~130,000 ops | Sorted all 10k points for 1 answer |
| **k << n** | Wasteful | Ranks all n when only k needed |

---

## 🚀 The Optimal Approach: $O(n \log k)$

Max-heap of size k on distance². Push new point as `(-dist², x, y)` (negated for max-heap). If heap exceeds k, heappop removes the farthest point.

### The Key Lifecycle Rule
1. **Push** `(-distance², x, y)` — negated so largest distance² becomes min-heap root
2. **If heap size > k:** heappop removes the farthest (most negative = farthest negated)
3. **Extract** all `[x, y]` from the heap at the end

---

## ✅ Mathematical Proof

n pushes, each O(log k). At most n pops (one per push when heap > k), each O(log k). Total: O(n log k). When k << n, this is significantly better than O(n log n).

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> The max-heap of size k is the "best k seen so far" filter — exactly what we need without sorting all n points.
</div>

## 🚀 Approach 2: Max-Heap of Size k — $O(n \log k)$
---

Instead of sorting all points, we use a **max-heap of size k** where root = farthest of the k closest.

As we iterate:
1. For each point, compute `dist² = x² + y²` (no sqrt — preserves ordering)
2. Push `(-dist², x, y)` — negated for max-heap simulation
3. If heap size > k: heappop removes the farthest point
4. Extract all `[x, y]` from the final heap

In [ ]:
import heapq

# Option A: Manual max-heap of size k — O(n log k)
def kClosest(points: list[list[int]], k: int) -> list[list[int]]:
    """
    Optimal: Max-Heap of size k
    Time: O(n log k) | Space: O(k)
    """
    heap = []
    for x, y in points:
        dist = -(x*x + y*y)   # negate for max-heap (most distant = largest dist² = min in negated heap)
        heapq.heappush(heap, (dist, x, y))
        if len(heap) > k:
            heapq.heappop(heap)   # evict the farthest of the top k+1
    return [[x, y] for (_, x, y) in heap]


# Option B: heapq.nsmallest — cleaner, same complexity
def kClosest_v2(points, k):
    """
    Optimal (cleaner): heapq.nsmallest with key
    Time: O(n log k) | Space: O(k)
    """
    return heapq.nsmallest(k, points, key=lambda p: p[0]**2 + p[1]**2)


print("Optimal:", kClosest([[1,3],[-2,2]], 1))          # Expected: [[-2, 2]]
print("Optimal:", kClosest([[3,3],[5,-1],[-2,4]], 2))   # Expected: [[3,3],[-2,4]]
print("Cleaner:", kClosest_v2([[1,3],[-2,2]], 1))       # Expected: [[-2, 2]]

## 🎤 5 Interview Q&A

### Q1: Why not compute the square root for Euclidean distance?

**Answer:** The square root function is monotone — if x² + y² < a² + b², then √(x²+y²) < √(a²+b²). This means the ordering of distances is identical whether we use d² or d. Since we only compare distances (never use the actual value), skipping sqrt saves computation and avoids floating-point precision issues.

---

### Q2: When would you use QuickSelect instead of a heap for "k closest"?

**Answer:** QuickSelect runs in O(n) average time but O(n²) worst case. Use QuickSelect when: (1) k ≈ n/2 (heap advantage shrinks since log k ≈ log n), (2) you need guaranteed O(n) average with acceptable worst case, (3) you want in-place without extra space. Use heap when: (1) k << n (log k << log n), (2) you need guaranteed O(n log k) without variance, (3) streaming data where points arrive one by one.

---

### Q3: What is the time complexity of the optimal approach and why?

**Answer:** O(n log k). We process each of the n points once. For each point: heappush is O(log k), and heappop (if heap > k) is O(log k). Both operations are on a heap that never exceeds k+1 elements, so O(log k) not O(log n). Total: n × O(log k) = O(n log k).

---

### Q4: What does `heapq.nsmallest` do internally?

**Answer:** For small k relative to n, `heapq.nsmallest(k, iterable, key)` uses a max-heap of size k internally — same algorithm as Option A. For large k, it falls back to sorting. Python chooses the better approach based on k vs n ratio. The manual Option A is equivalent but more explicit.

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) k = n — return all points (heap ends up with all n, same as sorting). (2) k = 1 — return the single closest point; heap size 1 throughout. (3) Duplicate distances — heap handles equal distances fine, keeps k of them. (4) Points at origin (0,0) — distance = 0, always closest, always in heap.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Euclidean Distance** | d = √(x² + y²) — straight-line distance from point (x,y) to origin |
| **Distance Squared** | d² = x² + y² — same ordering as d, avoids sqrt computation |
| **Max-Heap of Size k** | Maintains the k smallest elements seen so far; the root (maximum) is the eviction boundary |
| **Negation Trick** | Simulating a max-heap by negating keys — most negative = original maximum |
| **QuickSelect** | Partial sort algorithm — O(n) average to find kth smallest without fully sorting |
| **K Nearest Neighbors** | The problem of finding the k points closest to a query — fundamental in ML, spatial indexing, recommendation systems |
| **Eviction Threshold** | The root of the max-heap of size k — any element farther than this is immediately discarded |
| **nsmallest** | heapq.nsmallest(k, iterable) — returns k smallest using a max-heap internally |

## 💼 The Citi Narrative

**Context:** Citi's infrastructure team maintained regional data centers with servers distributed across geographic locations. For latency optimization, new applications needed to be routed to their k nearest data center nodes — minimizing round-trip time.

**Scenario:** Given 500 data center nodes with lat/lon coordinates, find the 3 nearest nodes to each of 10,000 application deployments. Naive: sort all 500 for each deployment = 10,000 × O(500 log 500) = 45 million sort operations. Max-heap: O(500 log 3) ≈ 8,000 operations per deployment.

**How this pattern applied:** For each deployment, the max-heap of size 3 scanned all 500 nodes in O(500 log 3) ≈ 7,900 comparisons. The heap root at each step was the "farthest acceptable node" — any node farther was immediately rejected. Final heap contained exactly the 3 nearest nodes.

**Impact:** Routing computation for 10,000 deployments completed in seconds instead of minutes. More importantly, the approach was streaming-friendly: as new data center nodes came online, they could be evaluated against the current k-nearest without rerunning the full search.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Given a list of points and a query point (not origin),
# find the k closest points to the QUERY POINT (not origin).
# -------------------------------------------------------

import heapq

def kClosestToQuery(points, query, k):
    # Compute distance² from each point to query (qx, qy)
    # Use max-heap of size k
    qx, qy = query
    # Your solution here
    pass


# Test
points = [[1,3],[-2,2],[5,5],[0,0]]
print(kClosestToQuery(points, [1,1], 2))   # Expected: [[1,3],[0,0]] (or [[0,0],[1,3]])

## 🎯 Summary: Key Takeaways

### The Pattern
**Max-Heap of Size k** — Filter for k smallest without sorting all n; root = current farthest-acceptable boundary

### When to Use It
- ✅ K nearest neighbors / k smallest / k closest
- ✅ k << n (heap advantage over sort is large)
- ✅ Streaming data where points arrive one by one
- ❌ **Don't use when:** k ≈ n — just sort (heap overhead isn't worth it for large k)

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force (sort) | $O(n \log n)$ | $O(n)$ |
| Optimal (Max-Heap size k) | $O(n \log k)$ | $O(k)$ |

### Interview Confidence Checklist
- [ ] Can explain why d² (no sqrt) is correct for comparison
- [ ] Can explain max-heap eviction boundary concept
- [ ] Can write both the manual and nsmallest versions
- [ ] Can compare to QuickSelect and state when to use each
- [ ] Can map to spatial routing use case with numbers

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master K Closest Points and you've unlocked the foundation of spatial indexing, K-nearest neighbors in ML, and network routing optimization. 🚀